# Stent and balloon

The stent is opened by contact with an inflating balloon, instead of being told where its
nodes must go. That difference is the reason this type exists. It tests the mechanism of a
real deployment, and it also tests the mixed-dimensional beam-to-solid contact that the
whole project is built around.

There is no artery here, so what is tested is the contact between the stent and the
balloon, and not the physics of a deployment into tissue. The modelling follows Datz et
al. (2025). The balloon is a solid tube with an orthotropic wall, stiff along its length
and soft around it, and that contrast is what makes a plain cylinder inflate like a folded
catheter balloon rather than stretching lengthwise.

This one takes hours, so it is built here and solved from a terminal.

In [ ]:
# "sphinx_gallery" renders each Plotly figure as a self-contained text/html output,
# so the views below also work on the documentation website without a running kernel.
import plotly.io as pio
pio.renderers.default = "sphinx_gallery"

## 1. Check the toolchain

4C runs inside a Docker container, so there is no 4C to install and nothing to compile.
What is needed is Docker itself, about 6 GB of free disk for the image, and Rosetta on
Apple Silicon, because the image is built for amd64 only.

`preflight()` checks all of that before a solve is started. A failing item is named
together with the command that fixes it.

In [ ]:
from stentfit.sim import FourCRunner, print_preflight

print_preflight(FourCRunner().preflight())

## 2. Load the stent

The stent has to be skeletonised already, which is what `stent_skeleton.ipynb` does.
Everything below is sized from the measurements in that output, so the stent is the
only input. A crimped design is closer to the real geometry of a stent
before deployment, so it suits this case better than an already-open one.

In [ ]:
from pathlib import Path

from stentfit import Simulation, Stent
from stentfit.sim import StentBalloonSettings

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

STENT_NAME = "stent04"
STENT_DIR = REPO / "examples/data/output/stent_skeleton" / STENT_NAME
OUTPUT_DIR = REPO / "examples/data/output/simulation"

stent = Stent.load(str(STENT_DIR), stent_name=STENT_NAME)

## 3. Parameters

Everything that can be changed sits in this one cell, grouped from the stent outwards: the
strut material first, then the balloon's shape and loading, then the contact between them,
then the two meshes and the solver.

The pressure is not a knob that is tuned until the diameter looks right. The diameter is
the outcome. Raising the balloon's own stiffness raises the pressure with it, which is what
makes a stiffness sweep mean anything.

In [ ]:
# 1. What to simulate
SB_BEAM_MATERIAL   = "elastic"          # "elastic" | "elastoplastic"
SB_BALLOON_MATERIAL = "orthotropic"     # "orthotropic" (the paper's) | "isotropic"
SB_LOAD_PROFILE    = "ramp_inflate_deflate"   # "ramp_inflate"             up, stays there
                                              # "ramp_inflate_deflate"     triangle (the paper)
                                              # "parabola_inflate_deflate" smooth, steep at

# 2. Stent material
# To restore the paper's values for a plastic run: SB_YOUNGS = 3.8e5 ; SB_YIELD_STRENGTH = 471.6
SB_YOUNGS          = 2.0e5          # MPa
SB_YIELD_STRENGTH  = 300.0          # MPa; unused while elastic
SB_STRUT_THICKNESS = stent.stent_features["strut_thickness"]   # mm; None = the measured value

# 3. Balloon shape
SB_CLEARANCE_FRAC  = 0.02           # gap between stent and balloon, as a fraction of strut thickness
SB_OVERHANG_FRAC   = 0.1            # how much longer the balloon is, as a fraction of stent length
SB_WALL            = 0.04           # balloon wall thickness in mm, from the paper
SB_RADIAL_STRAIN   = 0.50           # target radial strain, as a fraction of the undeformed radius

# 4. Balloon loading
SB_NEOHOOKE_YOUNGS = 17.0           # MPa; the wall's own stiffness
SB_END_SPRING      = 1000.0         # MPa/mm; the paper's end supports
SB_PRESSURE_MAX    = 0.6            # MPa

SB_NEOHOOKE_POISSON = 0.0           # exactly 0 stops expansion pulling the tube shorter
SB_FIBRE_LONG      = {"k1": 1000.0, "k2": 0.01}      # stiff along the tube
SB_FIBRE_CIRC      = {"k1": 1.5e-7, "k2": 0.35}      # soft around it
                                    # 10 orders of magnitude between them IS the mechanism

# 5. Contact between stent and balloon
SB_PENALTY         = 30.0           # N/mm^2, straight from Datz et al.
SB_PENALTY_LAW     = "linear"       # "linear" | "linear_quadratic"
SB_PENALTY_G0_PER_STRUT = 0.5       # contact easing-in width, as a multiple of
                                    # strut diameter. 0 = hard on/off switch
SB_DISCRETIZATION  = "gauss_point_to_segment"   # "mortar" | "gauss_point_to_segment"
SB_GAUSS_POINTS    = 6
SB_CONTACT_TYPE    = "gap_variation"
SB_MORTAR_SHAPE    = "line2"                      # mortar only
SB_MORTAR_DEFINED_IN = "reference_configuration"  # mortar only

# 6. Mesh sizes
SB_FACTOR_SOLID    = 1.5           # solid element / beam diameter        (limit: >= 1)
SB_FACTOR_BEAM     = 1.2           # beam element / solid element         (band: 1 .. 6)

# 7. Solver
SB_N_STEPS         = 50
if SB_BEAM_MATERIAL == "elastoplastic":
    SB_N_STEPS = SB_N_STEPS * 2
if SB_LOAD_PROFILE.endswith("_inflate_deflate"):
    SB_N_STEPS = SB_N_STEPS * 2

SB_MAX_ITER        = 60            # None -> 60
SB_TOL_RESIDUUM    = 1e-8          # None -> 1e-8
SB_TOL_INCREMENT   = 1e-10         # None -> 1e-10
SB_PREDICTOR       = "ConstDis"    # None -> "ConstDis"

This cell only moves the names above into the settings object the pipeline takes. The
balloon itself is built from those same settings, so there is one block to read rather
than two objects to keep in step.

In [ ]:
sb = StentBalloonSettings(
    material=SB_BEAM_MATERIAL,
    balloon_material=SB_BALLOON_MATERIAL, load_profile=SB_LOAD_PROFILE,
    youngs=SB_YOUNGS, yield_strength=SB_YIELD_STRENGTH, strut_thickness=SB_STRUT_THICKNESS,
    clearance_frac=SB_CLEARANCE_FRAC, overhang_frac=SB_OVERHANG_FRAC, wall=SB_WALL,
    radial_strain=SB_RADIAL_STRAIN,
    pressure_max=SB_PRESSURE_MAX,
    end_spring_stiffness=SB_END_SPRING,
    neohooke_youngs=SB_NEOHOOKE_YOUNGS, neohooke_poisson=SB_NEOHOOKE_POISSON,
    fibre_longitudinal=SB_FIBRE_LONG, fibre_circumferential=SB_FIBRE_CIRC,
    penalty=SB_PENALTY, penalty_law=SB_PENALTY_LAW, penalty_g0_per_strut=SB_PENALTY_G0_PER_STRUT,
    discretization=SB_DISCRETIZATION, gauss_points=SB_GAUSS_POINTS,
    contact_type=SB_CONTACT_TYPE, mortar_shape_function=SB_MORTAR_SHAPE,
    mortar_contact_defined_in=SB_MORTAR_DEFINED_IN,
    factor_solid=SB_FACTOR_SOLID, factor_beam=SB_FACTOR_BEAM,
    n_steps=SB_N_STEPS, max_iter=SB_MAX_ITER, tol_residuum=SB_TOL_RESIDUUM,
    tol_increment=SB_TOL_INCREMENT, predictor=SB_PREDICTOR)

print(sb.material, sb.balloon_material, f"penalty={sb.penalty:g}", sb.penalty_law,
      f"g0={sb.penalty_g0_per_strut:g}x strut", f"{sb.n_steps} steps")

## 4. Build the input

Each build gets its own numbered `runNNN` folder, so no two runs overwrite each other.
`check()` reports the beam-to-solid coupling, which this type does have.

Two parts of the output are worth reading. The balloon is placed against the stent's
measured innermost strut surface rather than against the averaged inner radius, because
how far the true innermost node sits inside that average changes from stent to stent, and
a clearance measured against the average would mean a different real gap on each one.

The other part is the restraint block. The stent is held by only the constraints that
remove its rigid-body modes, and each one has to act tangentially, because a constraint
with a radial component would fight the expansion that the contact is meant to drive. The
`% radial` column reports how much each one leaks. A crown that does not sit exactly where
the constraint wants it leaks a little, and that is the geometry of the ring rather than a
mistake.

In [ ]:
sim = Simulation(stent, sim_type="stent_balloon", settings=sb, output_dir=OUTPUT_DIR)
balloon = sim.balloon

written = sim.build_input()
sim.check()

for path in written:
    print(f"\n{path.name}")
    for f in sorted(path.parent.iterdir()):
        print(f"   {f.name:38s} {f.stat().st_size / 1e6:7.2f} MB")

## 5. Solve and measure

This one runs for hours, so it belongs in a terminal. Each run locks its own folder, so
several can be launched at the same time without colliding, and `--cpus` keeps them from
starving each other inside Docker's single VM.

`report` writes `metrics.csv`, `summary.yaml` and `balloon_profile.csv` next to the
results. The last one is the balloon radius along its length at six pressures, which is
the like-for-like comparison against Fig. 5 of Datz et al.

If the balloon does not inflate correctly on its own, no coupled result built on it means
anything, and `balloon.verify()` checks that with no beams and no contact search, which is
much cheaper than a coupled run.

In [ ]:
run = sim.built[0]

print("run these in a terminal:\n")
print(f"  python -m stentfit.run solve  stent_balloon {STENT_NAME} {run['name']} --cpus 4")
print(f"  python -m stentfit.run report stent_balloon {STENT_NAME} {run['name']}")
print("\nor check the balloon on its own first, which is much cheaper:")
print("  balloon.verify()")

## 6. Comparing the runs

This is the tuning loop. Change a parameter in section 3, re-run from there, and every
build lands in its own folder. `runs_summary.csv` is rebuilt from the run records each
time, so it never drifts from what is on disk.

In [ ]:
import pandas as pd

index = OUTPUT_DIR / "stent_balloon" / STENT_NAME / "runs_summary.csv"
if index.exists():
    display(pd.read_csv(index))

Every run also records the parameters that produced it, so an earlier run can be rebuilt
without remembering what was typed.

In [ ]:
restored = Simulation.from_run(sim.built[0]["run_dir"], stent=stent)
print(restored)
print(f"  material  {restored.settings.material}, E = {restored.settings.youngs:g} MPa")
print(f"  loading   {restored.settings.pressure_max:g} MPa, {restored.settings.load_profile}")
print(f"  solver    {restored.settings.n_steps} steps, predictor {restored.settings.predictor}")

## 7. Where the files are

```
examples/data/output/simulation/stent_balloon/<stent>/
    runs_summary.csv             one row per run, rebuilt from the records
    runNNN/
        balloon.4C.yaml          the balloon on its own
        stent_balloon.4C.yaml    the input 4C solves
        run_parameters.yaml      every parameter that produced this run
        run.log                  the solver output
        out_*/                   4C's own results
        results/metrics.csv      one row per load step
        results/summary.yaml     the headline numbers
        results/balloon_profile.csv   balloon radius along its length, at six pressures
```

In ParaView, open `out_*-structure-beams.pvd` for the stent and
`out_*-structure.pvd` for the balloon. Do not warp the beams, because their points are
already deformed. Add **Extract Surface** and then **Tube** to see the struts with their
real thickness.